In [1]:
import nest_asyncio
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai import capture_run_messages, AgentRunResult
import json
from pathlib import Path
import datetime
from dataclasses import dataclass

from pydantic_evals import Case, Dataset
from pydantic_ai.settings import ModelSettings
from pydantic_evals.evaluators import LLMJudge


nest_asyncio.apply()
load_dotenv()

def extract_attrs_from_instance(instance):
    result = {}
    for attribute in instance.__dict__:
        result[attribute] = getattr(instance, attribute)

    return result

def report_to_json_serializable(report):
    from copy import deepcopy
    report = deepcopy(report)

    cases = [
            {
                "case_name": case.name,
                "case_task_duration": case.task_duration,
                "case_total_duration": case.total_duration,
                "case_scores": [extract_attrs_from_instance(evaluation_result) for _, evaluation_result in case.scores.items()]
            }
            for case in report.cases
        ]

    # clean scores to be json serializable. Remove the source since it contains the EvaluatorSpec instance which is not needed
    for case_data in cases:
        case_data["case_scores"] = [
            {k: v for k, v in score.items() if k != "source"}
            for score in case_data["case_scores"]
        ]


    result = {
        "report_name": report.name,
        "cases": cases
    }
    
    return result

In [2]:
BASE_PATH = Path.cwd()
DATA_FILE_PATH = BASE_PATH.joinpath(
    "llm_responses_all_results_with_metrics_judged_FINAL.json"
)
REPORTS_DIR_BASE_PATH = BASE_PATH.joinpath("reports")
REPORTS_DIR_BASE_PATH.mkdir(exist_ok=True, parents=True)

sistemas_a_evaluar = [
    "llm_responses_no_rag",
    "llm_responses_base_rag",
    "llm_responses_agentic_rag",
]

modelos_usados_en_respuestas = ['gpt-5-nano', 'gpt-5-mini', 'gpt-5']

evaluators = []

class RubricaEvaluacion(BaseModel):
    nombre_evaluacion: str
    descripcion: str
    criterios: dict[str, str]

# modelos = ["ollama:gpt-oss:120b-cloud"]
# modelos = ["ollama:qwen3:8b"]

modelo_y_settings = [
    ("openrouter:moonshotai/kimi-k2-0905", ModelSettings(temperature=0.0, max_tokens=5000)),
]

for modelo, settings in modelo_y_settings:
    evaluators.extend(
        [
            LLMJudge(
                rubric=RubricaEvaluacion(
                    nombre_evaluacion="respuesta_correcta",
                    descripcion="La respuesta provista está alineada con la respuesta esperada. (Evalúa en español)",
                    criterios={
                        "0.0": "La respuesta está en blanco o no coincide en absoluto con la respuesta esperada.",
                        "0.25": "La respuesta tiene poca relación con la respuesta esperada y contiene errores significativos.",
                        "0.5": "La respuesta es parcialmente correcta, pero omite información clave o tiene errores menores.",
                        "0.75": "La respuesta es mayormente correcta, con solo errores menores.",
                        "1.0": "La respuesta es completamente correcta y precisa.",
                    },
                ).model_dump(),
                score={"evaluation_name": "respuesta_correcta", "include_reason": True},
                include_input=True,
                include_expected_output=True,
                model=modelo,
                model_settings=settings,
            ),
            LLMJudge(
                rubric=RubricaEvaluacion(
                    nombre_evaluacion="precision_legal",
                    descripcion="La respuesta cita correctamente los artículos relevantes del Código de Minería Argentino. (Evalúa en español)",
                    criterios={
                        "0.0": "No cita ningún artículo relevante o cita artículos incorrectos.",
                        "0.25": "Cita algunos artículos relevantes, pero omite otros importantes.",
                        "0.5": "Cita la mayoría de los artículos relevantes y otros irrelevantes.",
                        "0.75": "Cita casi todos los artículos relevantes con precisión y pocos irrelevantes.",
                        "1.0": "Cita todos los artículos relevantes y ninguno irrelevante de manera precisa y completa.",
                    },
                ).model_dump(),
                score={"evaluation_name": "precision_legal", "include_reason": True},
                assertion=False,
                include_input=True,
                include_expected_output=True,
                model=modelo,
                model_settings=settings,
            ),
        ]
    )

datasets = {}

#####################################
# Cases generation
#####################################

cases = []

preguntas = json.loads(DATA_FILE_PATH.read_text())

# Generate cases. 1 Case per question
for i, pregunta in enumerate(preguntas, start=1):
    case_input = pregunta["texto"]
    case_expected_output = pregunta["respuesta"]
    case_metadata = {
        "articulos_a_citar": ", ".join(pregunta["contexto_para_respuesta"]),
    }

    indice_pregunta = f"I{i:03}"
    nro_pregunta = f"P{pregunta['numero_pregunta']:02}"
    nro_subpregunta = f"SP{pregunta['numero_subpregunta']:02}" if pregunta['numero_subpregunta'] else "SP00"
    
    case_name = f"{indice_pregunta}-{nro_pregunta}-{nro_subpregunta}"

    case_ = Case(
        name=case_name,
        inputs=case_input,
        expected_output=case_expected_output,
        metadata=case_metadata,
    )

    cases.append(case_)

#####################################
# Question response mapping
#####################################

# Since I already generated the responses from the models I create a mapping between question and responses.
# In reality this would be populated with on the fly outputs when running a task.
# The mapping will be used in the task function when running a dataset
case_outputs = {}

for modelo_respuesta in modelos_usados_en_respuestas:
    for sistema in sistemas_a_evaluar:
        case_outputs[f"{modelo_respuesta}|{sistema}"] = []
        for pregunta in preguntas:
            for respuesta in pregunta[sistema]:
                if sistema == "llm_responses_agentic_rag":
                    response_model = respuesta["metadata"]["law_specialist_agent"]["usage"]["cost"]["model_id"]
                else:
                    response_model = respuesta["metadata"]["usage"]["cost"]["model_id"]
                if response_model == modelo_respuesta:
                    case_outputs[f"{modelo_respuesta}|{sistema}"].append((pregunta["texto"], respuesta["texto"]))

#####################################
# Dataset creation
#####################################

datasets = {}

for dataset_name in case_outputs.keys():
    datasets[dataset_name] = Dataset(
        name=dataset_name,
        cases=cases,
        evaluators=evaluators,
    )


In [3]:
#####################################
# RUN CODE
#####################################

results = []

for dataset_name, dataset in datasets.items():
    evaluator_model_data = dataset.evaluators[0].model.replace(":","_").replace("/", "_").replace(".", "-") # Asumes all evaluators use the same model.
    dataset_dict = dataset.model_dump()
    dataset_dict["start_timestamp"] = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    print(f'{dataset_name} - Start time: {dataset_dict["start_timestamp"]}')
    def contestar_pregunta(texto_pregunta: str) -> str: # Dynamically set the tasks for the dataset
        respuesta = next(filter(lambda x: x[0] == texto_pregunta, case_outputs[dataset_name]), None)
        return  respuesta[1] if respuesta else ""

    report = dataset.evaluate_sync(task=contestar_pregunta, progress=False, max_concurrency=5)
    dataset_dict["report"] = report_to_json_serializable(report)
    dataset_dict["end_timestamp"] = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    print(f'{dataset_name} - End time: {dataset_dict["end_timestamp"]}')

    REPORT_DIR_PATH = REPORTS_DIR_BASE_PATH.joinpath(evaluator_model_data)
    REPORT_DIR_PATH.mkdir(exist_ok=True, parents=True)

    with open(REPORT_DIR_PATH.joinpath(f"{dataset_dict['start_timestamp']}_report_{dataset_name.replace('|', '_')}.json"), "w", encoding="utf-8") as f:
        json.dump(dataset_dict, f, indent=4, ensure_ascii=False)

gpt-5-nano|llm_responses_no_rag - Start time: 20251123_161702
gpt-5-nano|llm_responses_no_rag - End time: 20251123_162413
gpt-5-nano|llm_responses_base_rag - Start time: 20251123_162413
gpt-5-nano|llm_responses_base_rag - End time: 20251123_163137
gpt-5-nano|llm_responses_agentic_rag - Start time: 20251123_163137
gpt-5-nano|llm_responses_agentic_rag - End time: 20251123_163633
gpt-5-mini|llm_responses_no_rag - Start time: 20251123_163633
gpt-5-mini|llm_responses_no_rag - End time: 20251123_164242
gpt-5-mini|llm_responses_base_rag - Start time: 20251123_164242
gpt-5-mini|llm_responses_base_rag - End time: 20251123_164951
gpt-5-mini|llm_responses_agentic_rag - Start time: 20251123_164951
gpt-5-mini|llm_responses_agentic_rag - End time: 20251123_165713
gpt-5|llm_responses_no_rag - Start time: 20251123_165713
gpt-5|llm_responses_no_rag - End time: 20251123_170539
gpt-5|llm_responses_base_rag - Start time: 20251123_170539
gpt-5|llm_responses_base_rag - End time: 20251123_171351
gpt-5|llm_re